In [3]:
import os
import requests
import psycopg2
from datetime import date
from dotenv import load_dotenv
import subprocess
import time
from tqdm import tqdm

# =========================
# Setup inicial
# =========================

load_dotenv()
caffeinate_process = subprocess.Popen(['caffeinate'])

dbname = os.getenv("POSTGRES_DB")
user = os.getenv("POSTGRES_USER")
password = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_HOST")
port = os.getenv("POSTGRES_PORT")
API_KEY = os.getenv("FMP_API_KEY")

conn = psycopg2.connect(
    dbname=dbname,
    user=user,
    password=password,
    host=host,
    port=port
)
cursor = conn.cursor()

HOY = date.today()

# =========================
# Obtener tickers base
# =========================

def obtener_tickers():
    cursor.execute("""
        SELECT ticker
        FROM api_raw.all_usa_ticker
    """)
    return [r[0] for r in cursor.fetchall()]

# =========================
# Obtener profile por ticker
# =========================

def obtener_profile(ticker):
    url = f"https://financialmodelingprep.com/api/v3/profile/{ticker}"
    params = {"apikey": API_KEY}

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()

    data = r.json()
    if not data:
        return None

    p = data[0]

    # Validación defensiva opcional
    if p.get("symbol") and p["symbol"] != ticker:
        tqdm.write(f"⚠️ Mismatch ticker {ticker} vs API {p['symbol']}")

    return (
        ticker,
        p.get("companyName"),
        p.get("cik"),
        p.get("exchangeShortName"),
        p.get("sector"),
        p.get("industry"),
        p.get("country"),
        p.get("isAdr"),
        p.get("isActivelyTrading"),
        HOY,
        "FMP"
    )

# =========================
# Insertar batch en DB
# =========================

def insertar_batch(batch):
    sql = """
        INSERT INTO api_raw.company_profile (
            ticker,
            companyname,
            cik,
            exchange_short_name,
            sector,
            industry,
            country,
            is_adr,
            is_actively_trading,
            fecha,
            source
        )
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        ON CONFLICT (ticker, fecha) DO NOTHING;
    """
    cursor.executemany(sql, batch)
    conn.commit()

# =========================
# Proceso principal
# =========================

def ejecutar_ingesta():
    tickers = obtener_tickers()
    total = len(tickers)

    print(f"📥 Tickers a procesar: {total}")

    batch = []
    errores = 0

    for ticker in tqdm(tickers, desc="📊 Descargando company_profile", unit="ticker"):
        try:
            profile = obtener_profile(ticker)
            if profile:
                batch.append(profile)

            if len(batch) >= 100:
                insertar_batch(batch)
                batch.clear()

            time.sleep(0.21)  # ⬅️ Rate limit (≈285 req/min)

        except Exception as e:
            errores += 1
            tqdm.write(f"⚠️ Error {ticker}: {e}")
            time.sleep(1)

    if batch:
        insertar_batch(batch)

    print(f"✅ Ingesta finalizada | Errores: {errores}")

# =========================
# Lanzar
# =========================

if __name__ == "__main__":
    try:
        ejecutar_ingesta()
    finally:
        caffeinate_process.terminate()
        cursor.close()
        conn.close()


📥 Tickers a procesar: 12050


📊 Descargando company_profile: 100%|█| 12050/12050 [3:13:09<00:00,  1.04ticker/

✅ Ingesta finalizada | Errores: 0
